[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-10-capstone-domain-model.ipynb#scrollTo=ca10b2c3)

---
# Day 10 · Capstone — Fine-Tune and Serve a Domain-Specific Llama Model
**certified-journeys / llama-certified** · Day 10 · Capstone Exam

> **Goal for today:** Fine-tune Llama-3.2-1B with QLoRA on a domain dataset, merge the adapter, convert to GGUF Q4_K_M, write a Modelfile, serve via Ollama's OpenAI-compatible endpoint, and benchmark fine-tuned vs base on 20 held-out prompts measuring accuracy and tokens/sec.


In [ ]:
%pip install -q transformers peft datasets trl bitsandbytes accelerate huggingface_hub openai psutil


## Capstone Architecture

This notebook implements the full end-to-end pipeline:

```
1. Domain dataset (Alpaca JSON)
        ↓
2. QLoRA fine-tuning  (SFTTrainer + BitsAndBytes 4-bit)
        ↓
3. Merge LoRA adapter → full model checkpoint (safetensors)
        ↓
4. GGUF conversion  (llama.cpp convert_hf_to_gguf.py + Q4_K_M quantisation)
        ↓
5. Ollama Modelfile  (domain system prompt + serving config)
        ↓
6. OpenAI-compatible REST API  (http://localhost:11434/v1)
        ↓
7. Benchmark: 20 held-out prompts → accuracy score (LLM judge) + tokens/sec
```

**Colab note:** Steps 2–6 require a GPU and local Ollama. All cells in this notebook run with mocks so you can study the pipeline; the Challenge cell provides the production scaffold.


## Step 1 · Choose a Domain Dataset in Alpaca Format

The Alpaca format is the most widely supported format for instruction fine-tuning:

```json
{"instruction": "...", "input": "...", "output": "..."}
```

| Domain | Example HuggingFace dataset | Rows |
|---|---|---|
| Customer support | `bitext/Bitext-customer-support-llm-chatbot-training-dataset` | 26 872 |
| Medical QA | `medalpaca/medical_meadow_medqa` | 10 178 |
| Code | `iamtarun/python_code_instructions_18k_alpaca` | 18 612 |
| Legal | `lex_glue` (adapt to Alpaca) | varies |

We use a synthetic customer-support dataset so the notebook runs without internet access.


In [ ]:
import json, random

# Synthetic domain dataset in Alpaca format — customer support
DOMAIN = 'customer_support'
ALPACA_DATASET = [
    {'instruction': 'How do I reset my password?', 'input': '', 'output': 'Go to Settings > Security > Reset Password. You will receive an email with a link.'},
    {'instruction': 'My order has not arrived. What should I do?', 'input': 'Order #12345', 'output': 'Please allow 2 business days for processing. Track your order at track.example.com/12345.'},
    {'instruction': 'How do I cancel my subscription?', 'input': '', 'output': 'Navigate to Account > Subscription > Cancel. Your access continues until the end of the billing period.'},
    {'instruction': 'I was charged twice for one order.', 'input': 'Order #98765', 'output': 'I apologise for the duplicate charge. I will initiate a refund for the second transaction within 3-5 business days.'},
    {'instruction': 'How do I update my billing address?', 'input': '', 'output': 'Go to Account > Billing > Edit Address. Changes take effect on your next invoice.'},
    {'instruction': 'Can I return a product after 30 days?', 'input': 'Product: headphones', 'output': 'Standard returns are within 30 days. For defective items, our warranty covers replacements for 12 months.'},
    {'instruction': 'How do I contact a human agent?', 'input': '', 'output': 'Click Live Chat on our website (Mon-Fri 9-5 EST) or call 1-800-EXAMPLE.'},
    {'instruction': 'What payment methods do you accept?', 'input': '', 'output': 'We accept Visa, Mastercard, PayPal, and Apple Pay. Cryptocurrency is not currently supported.'},
]

# Inflate to 100 synthetic examples for demonstration
random.seed(42)
full_dataset = (ALPACA_DATASET * 13)[:100]
random.shuffle(full_dataset)

# Train/eval split (80/20)
split_idx   = int(len(full_dataset) * 0.8)
train_data  = full_dataset[:split_idx]
eval_data   = full_dataset[split_idx:]

print(f'Domain           : {DOMAIN}')
print(f'Total examples   : {len(full_dataset)}')
print(f'Train split      : {len(train_data)}')
print(f'Eval split       : {len(eval_data)}')
print()
print('Sample record:')
print(json.dumps(train_data[0], indent=2))


**What just happened?**

- We created a synthetic customer-support dataset in Alpaca format with 100 examples.
- The 80/20 train/eval split is standard — use the eval split to monitor overfitting during training.
- **Production:** replace this with a HuggingFace dataset: `load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')`.
- Alpaca's `input` field is optional — leave it empty (`''`) for tasks without additional context.


## Step 2 · Format Data for SFTTrainer

SFTTrainer (TRL) expects a `text` field containing the full formatted prompt + response.  
Use Llama-3's native chat template for best results:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful customer support agent.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction}\n{input}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{output}
<|eot_id|>
```


In [ ]:
SYSTEM_PROMPT = 'You are a helpful, concise, and friendly customer support agent.'

def alpaca_to_llama3_chat(record: dict) -> str:
    """Convert an Alpaca record to Llama-3 chat template format."""
    user_content = record['instruction']
    if record.get('input'):
        user_content += f'\n{record["input"]}'

    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        f'{SYSTEM_PROMPT}\n'
        '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'{user_content}\n'
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n'
        f'{record["output"]}\n'
        '<|eot_id|>'
    )


# Format all training examples
formatted_train = [{'text': alpaca_to_llama3_chat(r)} for r in train_data]
formatted_eval  = [{'text': alpaca_to_llama3_chat(r)} for r in eval_data]

print('Formatted example:')
print(formatted_train[0]['text'])
print(f'\nAvg length: {sum(len(x["text"]) for x in formatted_train) // len(formatted_train)} chars')


**What just happened?**

- We applied Llama-3's special tokens (`<|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`) — using the wrong template causes garbled outputs.
- SFTTrainer with `dataset_text_field='text'` reads this directly and handles padding/truncation.
- **Common mistake:** forgetting `<|eot_id|>` after the assistant turn causes the model to generate endless text.


## Step 3 · QLoRA Fine-Tuning with SFTTrainer

QLoRA adds two innovations on top of standard LoRA:
1. **4-bit base model** via `BitsAndBytesConfig` — reduces VRAM from ~7 GB to ~2 GB for a 7B model
2. **Double quantisation** — quantises the quantisation constants themselves, saving ~0.4 bits/param

Key LoRA hyperparameters:

| Param | Typical value | Effect |
|---|---|---|
| `r` (rank) | 8–64 | Higher → more expressivity, more VRAM |
| `lora_alpha` | 2× rank | Scales the LoRA output; higher = stronger adaptation |
| `target_modules` | `q_proj, v_proj` | Which weight matrices to adapt |
| `lora_dropout` | 0.05 | Regularisation — reduce if dataset is large |


In [ ]:
# Production fine-tuning code — runs on a GPU with 8+ GB VRAM
# In Colab: Runtime > Change runtime type > T4 GPU
#
# We define and print the full config; actual training is mocked below.

from dataclasses import dataclass

@dataclass
class QLoRAConfig:
    # Model
    base_model:       str   = 'meta-llama/Llama-3.2-1B'
    # BitsAndBytes 4-bit
    load_in_4bit:     bool  = True
    bnb_4bit_quant_type: str = 'nf4'   # Normal Float 4 — better than fp4 for LLMs
    bnb_4bit_compute_dtype: str = 'bfloat16'
    bnb_use_double_quant: bool = True
    # LoRA
    lora_r:           int   = 16
    lora_alpha:       int   = 32   # 2x r
    lora_dropout:     float = 0.05
    target_modules:   tuple = ('q_proj', 'k_proj', 'v_proj', 'o_proj',
                               'gate_proj', 'up_proj', 'down_proj')
    # Training
    num_epochs:       int   = 3
    batch_size:       int   = 4
    grad_accum_steps: int   = 4   # effective batch = 16
    learning_rate:    float = 2e-4
    max_seq_length:   int   = 512
    output_dir:       str   = './checkpoints/llama-3.2-1b-customer-support'


cfg = QLoRAConfig()
print('QLoRA Configuration:')
for k, v in cfg.__dict__.items():
    print(f'  {k:<28}: {v}')

print()
print('Estimated VRAM usage:')
# Q4 NF4: ~0.5 GB/B params; 1B model = ~0.5 GB; LoRA adapters ~50 MB
model_vram  = 1.0 * 0.5   # GB
lora_vram   = 0.05         # GB
optim_vram  = 0.3          # GB (Adam optimizer states for LoRA params only)
activ_vram  = cfg.batch_size * 0.05  # rough estimate
total_vram  = model_vram + lora_vram + optim_vram + activ_vram
print(f'  Model weights (Q4)   : {model_vram:.2f} GB')
print(f'  LoRA adapters        : {lora_vram:.2f} GB')
print(f'  Optimizer (LoRA only): {optim_vram:.2f} GB')
print(f'  Activations          : {activ_vram:.2f} GB')
print(f'  Total estimated      : {total_vram:.2f} GB  (fits in T4 16 GB with room)')


**What just happened?**

- `nf4` (Normal Float 4) is the recommended quantisation type for QLoRA — it preserves more precision in the tails of the weight distribution vs. standard int4.
- `target_modules` covers all projection matrices in both the attention and MLP blocks — essential for full adaptation.
- **Effective batch size = batch_size × grad_accum_steps** — gradient accumulation lets you simulate large batches on a small GPU.
- The 1B model's total VRAM is well under 2 GB — a Colab T4 (16 GB) can train multiple epochs in minutes.


In [ ]:
# Mock SFTTrainer execution — shows what production code does without running on GPU
import time, random

class MockSFTTrainer:
    """Simulates SFTTrainer.train() output for demonstration."""
    def __init__(self, config: QLoRAConfig, n_train: int):
        self.config  = config
        self.n_train = n_train

    def train(self):
        print('Training Llama-3.2-1B with QLoRA...')
        steps_per_epoch = self.n_train // (self.config.batch_size * self.config.grad_accum_steps)
        history = []
        for epoch in range(1, self.config.num_epochs + 1):
            # Simulate decreasing loss over epochs
            base_loss = 2.5 - (epoch - 1) * 0.6
            for step in range(1, steps_per_epoch + 1):
                step_loss = base_loss + random.uniform(-0.15, 0.15)
                if step == steps_per_epoch:   # log at end of each epoch
                    history.append({'epoch': epoch, 'loss': round(step_loss, 4)})
                    print(f'  Epoch {epoch}/{self.config.num_epochs} — loss: {step_loss:.4f}')
                    time.sleep(0.1)  # simulate training time
        return history

    def save_model(self, path: str):
        print(f'  Adapter saved → {path}/adapter_model.safetensors')
        print(f'  Adapter config → {path}/adapter_config.json')


trainer = MockSFTTrainer(cfg, n_train=len(train_data))
history = trainer.train()
trainer.save_model(cfg.output_dir)

print()
print('Training complete.')
print(f'Loss at epoch 1: {history[0]["loss"]}  →  epoch {cfg.num_epochs}: {history[-1]["loss"]}')


**What just happened?**

- SFTTrainer logs loss at each step; declining loss confirms the adapter is learning the domain.
- The adapter is saved as `adapter_model.safetensors` — a two-file checkpoint (weights + config).
- **Overfitting signal:** if eval loss starts rising while train loss falls, stop early or reduce epochs.
- The base model is NOT modified — only the small LoRA matrices are saved (~50 MB for r=16).


## Step 4 · Merge the LoRA Adapter into the Base Model

GGUF conversion requires a **full weight checkpoint** — you cannot convert a base model + separate adapter directly.  
The merge step folds the LoRA delta matrices into the base weights:

```
W_merged = W_base + (B × A) × (alpha / r)
```

**Critical:** save in `safetensors` format, not `pytorch_model.bin` shards.


In [ ]:
import os

MERGED_DIR = './checkpoints/llama-3.2-1b-customer-support-merged'

# Production merge code:
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     'meta-llama/Llama-3.2-1B',
#     torch_dtype=torch.bfloat16,
#     device_map='cpu',   # merge on CPU to avoid OOM
# )
# model_with_adapter = PeftModel.from_pretrained(base_model, cfg.output_dir)
# merged = model_with_adapter.merge_and_unload()  # folds LoRA into base weights
# merged.save_pretrained(MERGED_DIR, safe_serialization=True)  # MUST use safetensors
# tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-1B')
# tokenizer.save_pretrained(MERGED_DIR)

# Simulate the merge output
print('Merging LoRA adapter into base model weights...')
print(f'  Base model   : meta-llama/Llama-3.2-1B')
print(f'  Adapter      : {cfg.output_dir}/adapter_model.safetensors')
print(f'  Operation    : W_merged = W_base + (B * A) * (alpha / r)')
print(f'  Output format: safetensors (REQUIRED for GGUF conversion)')
print()

# Simulated merged checkpoint files
merged_files = [
    'config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'model.safetensors',          # full merged weights — ~2.1 GB for 1B
]
print(f'Merged checkpoint saved to {MERGED_DIR}/')
for f in merged_files:
    print(f'  ✓ {f}')

print()
print('WARNING: If you see pytorch_model.bin shards, re-save with safe_serialization=True')
print('         GGUF conversion will fail on sharded pytorch_model.bin format')


**What just happened?**

- `merge_and_unload()` computes `W_merged = W_base + (B × A) × (lora_alpha / r)` for every adapted layer.
- `safe_serialization=True` writes a single `model.safetensors` file — what llama.cpp's conversion script expects.
- **Most common failure point:** saving with default HuggingFace settings produces sharded `pytorch_model.bin` files that the GGUF converter cannot read without extra flags.
- The merged model is the same size as the base (~2.1 GB for 1B in bfloat16).


## Step 5 · Convert to GGUF Q4_K_M

GGUF is the binary format used by llama.cpp and Ollama. Q4_K_M means:

| Component | Meaning |
|---|---|
| `Q4` | 4-bit integer quantisation |
| `_K` | K-quant family — mixed precision (some layers at Q5/Q6 for quality) |
| `_M` | Medium size variant — best quality/size tradeoff |

**Conversion pipeline:**
```bash
# Step A: Convert safetensors → GGUF (F16)
python llama.cpp/convert_hf_to_gguf.py ./merged-model \
    --outtype f16 --outfile model-f16.gguf

# Step B: Quantise F16 GGUF → Q4_K_M
./llama.cpp/quantize model-f16.gguf model-q4_k_m.gguf Q4_K_M
```


In [ ]:
import time

GGUF_F16_PATH = './models/llama-3.2-1b-customer-support-f16.gguf'
GGUF_Q4_PATH  = './models/llama-3.2-1b-customer-support-q4_k_m.gguf'

# Production commands (run in terminal, not Python):
CONVERT_CMD  = f'python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype f16 --outfile {GGUF_F16_PATH}'
QUANTIZE_CMD = f'./llama.cpp/quantize {GGUF_F16_PATH} {GGUF_Q4_PATH} Q4_K_M'

print('GGUF Conversion Pipeline')
print('=' * 55)
print()
print('Step A — Convert HuggingFace to GGUF (F16):')
print(f'  $ {CONVERT_CMD}')
print(f'  Output size: ~2.1 GB  (F16 = 2 bytes/param)')
print()
print('Step B — Quantise F16 → Q4_K_M:')
print(f'  $ {QUANTIZE_CMD}')
print(f'  Output size: ~0.7 GB  (Q4_K_M ≈ 4.5 bits/param avg)')
print()

# Simulate quantisation progress
import sys
layers = ['embed.weight', 'attn.q_proj', 'attn.k_proj', 'attn.v_proj', 'attn.o_proj',
          'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj', 'norm.weight', 'lm_head.weight']

print('Simulating quantisation...')
for i, layer in enumerate(layers):
    bits = 4.5 if 'proj' in layer else 16.0   # embed/head kept at higher precision
    print(f'  [{i+1:>2}/{len(layers)}] {layer:<25} {bits:.1f} bits')

print(f'\n✓ GGUF Q4_K_M written → {GGUF_Q4_PATH}')
print(f'  Size reduction: F16 (2.1 GB) → Q4_K_M (0.7 GB) = 67% smaller')


**What just happened?**

- `convert_hf_to_gguf.py` reads the `model.safetensors` file and writes a single GGUF binary.
- The `quantize` tool applies Q4_K_M: most layers at 4-bit, attention and embedding layers at 6 or 8-bit for quality.
- **67% size reduction** (2.1 GB → 0.7 GB) with minimal quality loss — this is why GGUF is the production deployment format.
- The first and last layers (embedding + lm_head) are kept at higher precision — they have the most impact on output quality.


## Step 6 · Write a Modelfile and Load into Ollama

A Modelfile is Ollama's build specification — analogous to a Dockerfile:

```dockerfile
FROM ./path/to/model.gguf          # or a pulled model name
PARAMETER temperature 0.1           # low temp = more deterministic (good for support)
PARAMETER top_p 0.9
PARAMETER num_ctx 2048              # context window
SYSTEM """[system prompt]"""
```

After writing the Modelfile:
```bash
ollama create customer-support -f ./Modelfile
ollama run customer-support
```


In [ ]:
MODELFILE_CONTENT = f'''FROM {GGUF_Q4_PATH}

# Inference parameters — tuned for customer support (concise, factual)
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER num_predict 256

# Domain-specific system prompt
SYSTEM """
You are a helpful, concise, and professional customer support agent.
Always respond in 2-4 sentences.
If you cannot resolve an issue, direct the customer to the human support team.
Do not speculate about policies you are unsure of.
"""

# Template matching Llama-3 chat format
TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>
{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
'''

MODELFILE_PATH = './Modelfile'

# Write the Modelfile
with open(MODELFILE_PATH, 'w') as f:
    f.write(MODELFILE_CONTENT)

print('Modelfile written:')
print('-' * 55)
print(MODELFILE_CONTENT)

print('Next steps (run in terminal):')
print('  ollama create customer-support -f ./Modelfile')
print('  ollama list   # verify the model appears')
print('  ollama run customer-support')


**What just happened?**

- `temperature 0.1` makes the model deterministic and factual — ideal for support where hallucinations are costly.
- The `TEMPLATE` block must exactly match Llama-3's special tokens, or the model will ignore the system prompt.
- `num_ctx 2048` reserves KV cache for 2048 tokens — increase to 4096 for long support threads at the cost of RAM.
- `ollama create` builds a local model registry entry; `ollama push` can publish it to ollama.com.


## Step 7 · Serve via the OpenAI-Compatible Endpoint

After `ollama create`, the model is accessible via the same OpenAI SDK we used on Day 8.

This is the payoff of the GGUF pipeline: your fine-tuned domain model slots into **any OpenAI-compatible application** without code changes.


In [ ]:
import json, time, unittest.mock as mock

MODEL_NAME = 'customer-support'  # name given to ollama create

# Mock Ollama responses for Colab (swap base_url for real inference)
SUPPORT_RESPONSES = [
    'To reset your password, go to Settings > Security > Reset Password. You will receive an email within 5 minutes.',
    'I apologise for the inconvenience. Please allow 2 business days for your order to arrive. Track it at track.example.com.',
    'To cancel your subscription, go to Account > Subscription > Cancel. Access continues until the end of the billing period.',
]

_response_idx = [0]
def _mock_support_response(url, **kwargs):
    body = kwargs.get('json', {})
    resp = mock.MagicMock()
    resp.status_code = 200
    content = SUPPORT_RESPONSES[_response_idx[0] % len(SUPPORT_RESPONSES)]
    _response_idx[0] += 1
    resp.json.return_value = {
        'id': f'chatcmpl-{_response_idx[0]}',
        'model': MODEL_NAME,
        'choices': [{'message': {'role': 'assistant', 'content': content}, 'finish_reason': 'stop'}],
        'usage': {'prompt_tokens': 40, 'completion_tokens': 30, 'total_tokens': 70}
    }
    return resp


# Test queries against our fine-tuned customer support model
TEST_QUERIES = [
    'How do I reset my password?',
    'My order has not arrived after 5 days.',
    'I want to cancel my subscription immediately.',
]

print(f'Testing model: {MODEL_NAME}')
print(f'Endpoint:      http://localhost:11434/v1')
print('=' * 60)

with mock.patch('requests.Session.post', side_effect=_mock_support_response):
    from openai import OpenAI
    client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

    for i, query in enumerate(TEST_QUERIES, 1):
        # Simulate direct call
        response_data = _mock_support_response('', json={'messages': [{'role': 'user', 'content': query}]})
        result = response_data.json()
        reply = result['choices'][0]['message']['content']
        print(f'\nQ{i}: {query}')
        print(f'A:  {reply}')
        print(f'    ({result["usage"]["completion_tokens"]} tokens)')


**What just happened?**

- The fine-tuned model answers domain questions with the concise, professional tone defined in the system prompt.
- The OpenAI client doesn't know it's talking to a locally-served GGUF file — the API contract is identical.
- **Integration pattern:** swap `base_url` at the environment variable level — your FastAPI proxy, Langchain chain, or chat UI work unchanged.


## Step 8 · Build the 20-Prompt Held-Out Benchmark Set

The held-out set must **not** overlap with training data — otherwise we are measuring memorisation, not generalisation.


In [ ]:
# 20 held-out customer-support prompts NOT in the training set
HELD_OUT_PROMPTS = [
    {'id': 1,  'question': 'How do I change my email address on my account?'},
    {'id': 2,  'question': 'Can I get a refund if the product is opened?'},
    {'id': 3,  'question': 'How long does standard shipping take?'},
    {'id': 4,  'question': 'My account was locked. How do I unlock it?'},
    {'id': 5,  'question': 'Do you offer student discounts?'},
    {'id': 6,  'question': 'How do I add a second delivery address?'},
    {'id': 7,  'question': 'What is your privacy policy on user data?'},
    {'id': 8,  'question': 'I never received my order confirmation email.'},
    {'id': 9,  'question': 'Can I change the delivery address after placing an order?'},
    {'id': 10, 'question': 'How do I enable two-factor authentication?'},
    {'id': 11, 'question': 'My promo code is not working at checkout.'},
    {'id': 12, 'question': 'How do I download my purchase receipt?'},
    {'id': 13, 'question': 'Is there a loyalty rewards program?'},
    {'id': 14, 'question': 'How do I report a defective product?'},
    {'id': 15, 'question': 'Can I split payment across two cards?'},
    {'id': 16, 'question': 'What happens to my data if I delete my account?'},
    {'id': 17, 'question': 'How do I track a return shipment?'},
    {'id': 18, 'question': 'Do you ship internationally?'},
    {'id': 19, 'question': 'How do I upgrade my subscription plan?'},
    {'id': 20, 'question': 'What are your customer support hours?'},
]

assert len(HELD_OUT_PROMPTS) == 20
print(f'Held-out benchmark: {len(HELD_OUT_PROMPTS)} prompts')
print(f'Training overlap check: {len(set(p["question"] for p in HELD_OUT_PROMPTS) & set(r["instruction"] for r in train_data))} shared prompts (should be 0)')
print()
for p in HELD_OUT_PROMPTS[:5]:
    print(f'  {p["id"]:>2}. {p["question"]}')
print(f'  ... ({len(HELD_OUT_PROMPTS) - 5} more)')


**What just happened?**

- We verified zero overlap between held-out and training sets — the benchmark measures generalisation.
- 20 prompts is the minimum for statistically meaningful results; 50+ is preferred for production decisions.
- **Prompt diversity matters:** cover edge cases (locked accounts, international shipping) not just common flows (password reset).


## Step 9 · Run the Full Benchmark: Accuracy + Tokens/sec


In [ ]:
import random, statistics, time

def mock_model_inference(question: str, variant: str, delay_s: float = 0.05) -> dict:
    """Simulate inference for base or fine-tuned model."""
    time.sleep(delay_s)  # simulate decode time
    output_tokens = random.randint(20, 60)
    content = (
        f'[Fine-tuned response to: {question[:30]}...]'
        if variant == 'ft'
        else f'[Base response to: {question[:30]}...]'
    )
    return {
        'content':       content,
        'output_tokens': output_tokens,
        'latency_s':     delay_s + random.uniform(0.01, 0.03),
    }


def llm_judge_score(question: str, response: str, variant: str) -> int:
    """Mock judge — fine-tuned scores systematically higher on domain prompts."""
    if variant == 'ft':
        return random.choices([3, 4, 5], weights=[0.1, 0.4, 0.5])[0]
    else:
        return random.choices([2, 3, 4], weights=[0.2, 0.5, 0.3])[0]


def run_benchmark(variant: str, prompts: list[dict]) -> dict:
    """Run the full 20-prompt benchmark for one model variant."""
    delay = 0.04 if variant == 'ft' else 0.05   # ft Q4 slightly faster
    scores, latencies, token_counts = [], [], []

    for p in prompts:
        result = mock_model_inference(p['question'], variant, delay_s=delay)
        score  = llm_judge_score(p['question'], result['content'], variant)
        scores.append(score)
        latencies.append(result['latency_s'])
        token_counts.append(result['output_tokens'])

    total_tokens = sum(token_counts)
    total_time   = sum(latencies)
    return {
        'variant':       variant,
        'avg_score':     round(statistics.mean(scores), 2),
        'pct_ge4':       round(sum(1 for s in scores if s >= 4) / len(scores) * 100, 1),
        'avg_tps':       round(total_tokens / total_time, 1),
        'avg_latency_ms': round(statistics.mean(latencies) * 1000, 1),
        'scores':        scores,
    }


print('Running 20-prompt benchmark...')
base_bm = run_benchmark('base', HELD_OUT_PROMPTS)
ft_bm   = run_benchmark('ft',   HELD_OUT_PROMPTS)

print()
print('Benchmark Results — 20 held-out prompts')
print(f'{"Metric":<22} {"Base":>10} {"Fine-Tuned":>12} {"Delta":>10}')
print('-' * 57)
for key in ('avg_score', 'pct_ge4', 'avg_tps', 'avg_latency_ms'):
    b, f = base_bm[key], ft_bm[key]
    delta = f - b
    sign  = '+' if delta > 0 else ''
    print(f'{key:<22} {b:>10} {f:>12} {sign}{delta:>9.2f}')


**What just happened?**

- The fine-tuned model scores higher on domain prompts (higher avg_score, higher pct_ge4) — the training data generalised.
- Speed is comparable — fine-tuning changes weights but not architecture or quantisation level.
- **Interpret the delta carefully:** a +0.5 quality score improvement over 20 prompts is meaningful; a +0.1 improvement may be noise.
- Always report both `avg_score` AND `pct_ge4` — the latter is the operational metric (fraction of responses good enough to ship).


In [ ]:
# ============================================================
# CAPSTONE CHALLENGE
# ============================================================
# Implement the complete fine-tune-serve-benchmark pipeline end-to-end.
#
# Your pipeline must:
#   1. DATASET: Load a domain dataset in Alpaca format (customer support, medical QA,
#      code, or legal). Use HuggingFace datasets or a local JSON file.
#      Split into 80% train / 20% eval.
#
#   2. FINE-TUNE: Configure and run SFTTrainer with QLoRA:
#      - BitsAndBytesConfig: load_in_4bit=True, bnb_4bit_quant_type='nf4'
#      - LoraConfig: r=16, lora_alpha=32, target all projection layers
#      - TrainingArguments: 3 epochs, log loss per epoch
#      - Print loss curve: epoch 1 → epoch 3
#
#   3. MERGE: Use PeftModel.from_pretrained + merge_and_unload()
#      - Save merged model with safe_serialization=True
#      - Verify model.safetensors exists (not pytorch_model.bin)
#
#   4. CONVERT: Run llama.cpp conversion + Q4_K_M quantisation
#      - Step A: python convert_hf_to_gguf.py --outtype f16
#      - Step B: ./quantize model-f16.gguf model-q4_k_m.gguf Q4_K_M
#      - Print file sizes at each step
#
#   5. MODELFILE: Write a Modelfile with:
#      - FROM pointing to your GGUF file
#      - PARAMETER temperature 0.1 (or appropriate for your domain)
#      - SYSTEM prompt tuned to your domain
#      - TEMPLATE matching your base model's chat format
#      Run: ollama create <your-model-name> -f Modelfile
#
#   6. SERVE: Use the OpenAI Python SDK with base_url='http://localhost:11434/v1'
#      - Send 3 test queries to your fine-tuned model
#      - Print each question and response
#
#   7. BENCHMARK: Run the 20 held-out prompts against BOTH base and fine-tuned model
#      - Measure: LLM judge score (1-5) per prompt
#      - Measure: decode tokens/sec per prompt (time the call, divide by output tokens)
#      - Compute: avg_score, pct_ge4, avg_tps, delta_score (ft - base)
#      - Print a formatted summary table
#      - Write the results to benchmark_results.json
#
# Scaffold (replace each TODO with real code):

# --- 1. Dataset ---
# dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
# train_data = [alpaca_to_llama3_chat(r) for r in dataset['train']]
# TODO: load, format, split

# --- 2. Fine-tune ---
# bnb_config  = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
#                                   bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
# lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','k_proj','v_proj','o_proj',
#                                                                 'gate_proj','up_proj','down_proj'],
#                           lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
# TODO: load base model with bnb_config, wrap with get_peft_model(model, lora_config)
# TODO: create SFTTrainer and call trainer.train()
# TODO: print loss per epoch

# --- 3. Merge ---
# merged = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
# merged.save_pretrained(MERGED_DIR, safe_serialization=True)
# TODO: verify model.safetensors exists

# --- 4. Convert ---
# import subprocess
# subprocess.run(['python', 'llama.cpp/convert_hf_to_gguf.py', MERGED_DIR, '--outtype', 'f16', '--outfile', GGUF_F16_PATH])
# subprocess.run(['./llama.cpp/quantize', GGUF_F16_PATH, GGUF_Q4_PATH, 'Q4_K_M'])
# TODO: print file sizes at each step

# --- 5. Modelfile ---
# TODO: write Modelfile and run: subprocess.run(['ollama', 'create', MODEL_NAME, '-f', 'Modelfile'])

# --- 6. Serve ---
# client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# TODO: send 3 test queries and print responses

# --- 7. Benchmark ---
# TODO: implement run_full_benchmark(model_name, held_out_prompts)
# TODO: compare base vs fine-tuned, print table, write benchmark_results.json

print('Implement the 7 pipeline steps above.')
print('When complete, you will have:')
print('  - A domain-fine-tuned GGUF model served via Ollama')
print('  - A 20-prompt benchmark comparing base vs fine-tuned')
print('  - A benchmark_results.json file you can include in your portfolio')


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Alpaca format | `{instruction, input, output}` — most instruction datasets use this schema |
| QLoRA merge | `merge_and_unload()` folds LoRA deltas into base weights; save with `safe_serialization=True` |
| GGUF conversion | Two steps: HF → F16 GGUF, then quantise to Q4_K_M; safetensors input required |
| Modelfile | Sets temperature, context length, and system prompt; TEMPLATE must match base model's token format |
| OpenAI endpoint | `base_url='http://localhost:11434/v1'` and `api_key='ollama'` — zero other SDK changes |
| Benchmark | Report avg_score, pct_ge4, avg_tps, and catastrophic-forgetting check |

> **Tip:** The most common failure point is the GGUF conversion step — ensure the merged model is saved in safetensors format before converting, not pytorch_model.bin shards.

---
## Congratulations — you have completed the course!

You have built the full Llama production pipeline:
- Fine-tuned a domain-specific model with QLoRA
- Quantised it to GGUF Q4_K_M for efficient serving
- Served it via Ollama's OpenAI-compatible API
- Benchmarked it against the base model on 20 held-out prompts

Mark Day 10 complete in your [tracker](../index.html).
